# Generate a reviewed science visual bank on a Colab GPU
Use Runtime -> Change runtime type -> GPU. Free GPU availability and session duration are not guaranteed.
This notebook is interactive batch generation, not a scheduled API server. It uses Wan 2.1 1.3B,
a smaller option than Wan 2.2 5B/14B or Mochi, for memory-limited free hardware.
No hosted generation API key is required. Model download, RAM, disk and CUDA memory are still required.
The notebook was structurally checked; GPU inference has not been tested in this repository.
Generated science visuals are illustrations and must be reviewed for plausibility.


In [ ]:
%pip -q install "diffusers>=0.35.1,<1" "transformers>=4.45,<5" "accelerate>=1,<2" imageio imageio-ffmpeg sentencepiece


In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
from pathlib import Path
import json, shutil
assert torch.cuda.is_available(), 'Select a GPU runtime; do not attempt this on CPU.'
print(torch.cuda.get_device_name(0))
MODEL = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL, vae=vae, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()


In [ ]:
# Replace these with concrete visual subjects from your planned scripts.
SHOTS = [
    {'name': 'silicate-weathering', 'tags': ['silicate', 'weathering', 'rock', 'rainwater'],
     'prompt': 'Documentary illustration, close view of rainwater trickling across textured silicate rocks, slow camera push, natural overcast lighting, no text, no logos, no impossible dissolving rock.'},
]
# Start with one short low-resolution clip. Larger batches may exceed free runtime limits.
FRAME_COUNT = 49
HEIGHT, WIDTH = 480, 832
out = Path('/content/visual_bank'); out.mkdir(exist_ok=True)
manifest = {'clips': []}
for index, shot in enumerate(SHOTS):
    frames = pipe(prompt=shot['prompt'], height=HEIGHT, width=WIDTH,
                  num_frames=FRAME_COUNT, num_inference_steps=30, guidance_scale=5.0,
                  generator=torch.Generator(device='cpu').manual_seed(42 + index)).frames[0]
    filename = shot['name'] + '.mp4'
    export_to_video(frames, str(out / filename), fps=16)
    manifest['clips'].append({'file': filename, 'tags': shot['tags'], 'model': MODEL,
                              'approved': False, 'prompt': shot['prompt']})
(out / 'manifest.json').write_text(json.dumps(manifest, indent=2))


In [ ]:
from IPython.display import Video, display
for clip in manifest['clips']:
    print(clip['file'])
    display(Video(str(out / clip['file']), embed=True))


Review each clip above for wrong science, malformed objects, distracting motion and watermarks.
Only set approved=True for clips you reviewed. Tags must identify their actual visible content.
Publish the ZIP as a public GitHub Release asset and set repository variable VISUAL_BANK_URL to its download URL.
The workflow imports approved clips automatically and matches their tags against scene image cues.
It ignores unapproved clips; imported generated clips retain the AI-generated illustration label.


In [ ]:
# Edit individually after review, for example:
# manifest['clips'][0]['approved'] = True
(out / 'manifest.json').write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive('/content/visual-bank', 'zip', out)
from google.colab import files
files.download(archive)
